# RKD All-in-One (Teacher + Distillation) on Colab GPU

This notebook runs the full pipeline in one place:
1. Prepare Colab environment and GPU
2. Train and evaluate teacher
3. Distill student from teacher
4. Evaluate distilled student

In [ ]:
# Optional: mount Google Drive for persistent checkpoints
USE_DRIVE = False

if USE_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive")

In [ ]:
import os
import shlex
import subprocess
import sys
from pathlib import Path


def run_cmd(cmd):
    cmd = [str(x) for x in cmd]
    print("\n>>>", " ".join(shlex.quote(x) for x in cmd))
    subprocess.run(cmd, check=True)


REPO_URL = "https://github.com/Gabomfim/MO434.git"
REPO_ROOT = Path("/content/MO434")
RKD_ROOT = REPO_ROOT / "RKD"

if not REPO_ROOT.exists():
    run_cmd(["git", "clone", REPO_URL, str(REPO_ROOT)])
else:
    run_cmd(["git", "-C", str(REPO_ROOT), "pull"])

os.chdir(RKD_ROOT)
print("Working directory:", os.getcwd())

run_cmd(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "tqdm",
        "h5py",
        "scipy",
        "wandb",
        "kagglehub",
    ]
)

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "GPU is required. In Colab, go to Runtime > Change runtime type > GPU."
    )

print("GPU:", torch.cuda.get_device_name(0))
print("CUDA version:", torch.version.cuda)

## Configuration

In [ ]:
# Shared
DATASET = "cub200"  # cub200 | cars196 | stanford
DATA_DIR = "../data"

# W&B
WANDB_PROJECT = "rkd-metric-learning"
WANDB_ENTITY = ""  # optional
WANDB_MODE = "online"  # online | offline | disabled

# Teacher
TEACHER_BASE = "resnet50"
TEACHER_EMBEDDING_SIZE = "512"
TEACHER_L2NORMALIZE = "true"
TEACHER_SAVE_DIR = "teacher"
TEACHER_RUN_NAME = "teacher-r50-allinone"
TEACHER_GROUP = "teacher-runs"

# Teacher optimization
TEACHER_LR = "1e-5"
TEACHER_BATCH = "128"
TEACHER_EPOCHS = "40"
TEACHER_ITER_PER_EPOCH = "100"
TEACHER_LR_DECAY_EPOCHS = ["25", "30", "35"]
TEACHER_LR_DECAY_GAMMA = "0.5"
TEACHER_RECALL = ["1", "2", "4", "8"]

# Student
STUDENT_BASE = "resnet18"
STUDENT_EMBEDDING_SIZE = "64"
STUDENT_L2NORMALIZE = "false"
STUDENT_SAVE_DIR = "student"
STUDENT_RUN_NAME = "distill-r18-allinone"
STUDENT_GROUP = "distillation-experiments"

# Distillation loss weights
TRIPLET_RATIO = "0"
DIST_RATIO = "1"
ANGLE_RATIO = "2"
QUAD_RATIO = "0"
DARK_RATIO = "0"
DARK_ALPHA = "2"
DARK_BETA = "3"
AT_RATIO = "0"

# Distillation optimization
STUDENT_LR = "1e-4"
STUDENT_BATCH = "128"
STUDENT_EPOCHS = "80"
STUDENT_ITER_PER_EPOCH = "100"
STUDENT_LR_DECAY_EPOCHS = ["40", "60"]
STUDENT_LR_DECAY_GAMMA = "0.1"
STUDENT_RECALL = ["1", "2", "4", "8"]

# Distillation confusion matrix controls
LOG_CONFUSION_MATRIX = "true"
MAX_CONFUSION_CLASSES = "200"
MAX_CONFUSION_SAMPLES = "2000"

In [ ]:
from pathlib import Path
import kagglehub

if DATASET == "cub200":
    path = Path(kagglehub.dataset_download("wenewone/cub2002011"))
    data_root = path.parent if path.name == "CUB_200_2011" else path
    DATA_DIR = str(data_root)
    print("Path to dataset files:", str(path))
    print("Using --data:", DATA_DIR)
else:
    print(
        f"Skipping Kaggle download because DATASET={DATASET}. Using --data: {DATA_DIR}"
    )

## 1) Train Teacher

In [ ]:
import run as teacher_runner

teacher_train_params = {
    "mode": "train",
    "dataset": DATASET,
    "base": TEACHER_BASE,
    "sample": "distance",
    "loss": "l2_triplet",
    "margin": "0.2",
    "embedding_size": TEACHER_EMBEDDING_SIZE,
    "l2normalize": TEACHER_L2NORMALIZE,
    "lr": TEACHER_LR,
    "lr_decay_epochs": TEACHER_LR_DECAY_EPOCHS,
    "lr_decay_gamma": TEACHER_LR_DECAY_GAMMA,
    "batch": TEACHER_BATCH,
    "num_image_per_class": "5",
    "epochs": TEACHER_EPOCHS,
    "iter_per_epoch": TEACHER_ITER_PER_EPOCH,
    "recall": TEACHER_RECALL,
    "data": DATA_DIR,
    "save_dir": TEACHER_SAVE_DIR,
    "wandb_project": WANDB_PROJECT,
    "wandb_run_name": TEACHER_RUN_NAME,
    "wandb_group": TEACHER_GROUP,
    "wandb_mode": WANDB_MODE,
}

if WANDB_ENTITY.strip():
    teacher_train_params["wandb_entity"] = WANDB_ENTITY

teacher_runner.run_with_params(teacher_train_params)

## 2) Evaluate Teacher

In [ ]:
import run as teacher_runner

teacher_best_ckpt = f"{TEACHER_SAVE_DIR}/best.pth"

teacher_eval_params = {
    "mode": "eval",
    "dataset": DATASET,
    "base": TEACHER_BASE,
    "embedding_size": TEACHER_EMBEDDING_SIZE,
    "l2normalize": TEACHER_L2NORMALIZE,
    "batch": TEACHER_BATCH,
    "recall": TEACHER_RECALL,
    "load": teacher_best_ckpt,
    "data": DATA_DIR,
    "wandb_project": WANDB_PROJECT,
    "wandb_run_name": TEACHER_RUN_NAME + "-eval",
    "wandb_group": TEACHER_GROUP,
    "wandb_mode": WANDB_MODE,
}

if WANDB_ENTITY.strip():
    teacher_eval_params["wandb_entity"] = WANDB_ENTITY

teacher_runner.run_with_params(teacher_eval_params)

## 3) Distill Student

In [ ]:
import run_distill as distill_runner

distill_params = {
    "dataset": DATASET,
    "base": STUDENT_BASE,
    "teacher_base": TEACHER_BASE,
    "triplet_ratio": TRIPLET_RATIO,
    "dist_ratio": DIST_RATIO,
    "angle_ratio": ANGLE_RATIO,
    "quad_ratio": QUAD_RATIO,
    "dark_ratio": DARK_RATIO,
    "dark_alpha": DARK_ALPHA,
    "dark_beta": DARK_BETA,
    "at_ratio": AT_RATIO,
    "triplet_sample": "distance",
    "triplet_margin": "0.2",
    "l2normalize": STUDENT_L2NORMALIZE,
    "embedding_size": STUDENT_EMBEDDING_SIZE,
    "teacher_load": teacher_best_ckpt,
    "teacher_l2normalize": TEACHER_L2NORMALIZE,
    "teacher_embedding_size": TEACHER_EMBEDDING_SIZE,
    "lr": STUDENT_LR,
    "data": DATA_DIR,
    "epochs": STUDENT_EPOCHS,
    "batch": STUDENT_BATCH,
    "iter_per_epoch": STUDENT_ITER_PER_EPOCH,
    "lr_decay_epochs": STUDENT_LR_DECAY_EPOCHS,
    "lr_decay_gamma": STUDENT_LR_DECAY_GAMMA,
    "recall": STUDENT_RECALL,
    "log_confusion_matrix": LOG_CONFUSION_MATRIX,
    "max_confusion_classes": MAX_CONFUSION_CLASSES,
    "max_confusion_samples": MAX_CONFUSION_SAMPLES,
    "save_dir": STUDENT_SAVE_DIR,
    "wandb_project": WANDB_PROJECT,
    "wandb_run_name": STUDENT_RUN_NAME,
    "wandb_group": STUDENT_GROUP,
    "wandb_mode": WANDB_MODE,
}

if WANDB_ENTITY.strip():
    distill_params["wandb_entity"] = WANDB_ENTITY

distill_runner.run_with_params(distill_params)

## 4) Evaluate Student

In [ ]:
import run as teacher_runner

student_eval_params = {
    "mode": "eval",
    "dataset": DATASET,
    "base": STUDENT_BASE,
    "embedding_size": STUDENT_EMBEDDING_SIZE,
    "l2normalize": STUDENT_L2NORMALIZE,
    "batch": STUDENT_BATCH,
    "recall": STUDENT_RECALL,
    "load": f"{STUDENT_SAVE_DIR}/best.pth",
    "data": DATA_DIR,
    "wandb_project": WANDB_PROJECT,
    "wandb_run_name": STUDENT_RUN_NAME + "-eval",
    "wandb_group": STUDENT_GROUP,
    "wandb_mode": WANDB_MODE,
}

if WANDB_ENTITY.strip():
    student_eval_params["wandb_entity"] = WANDB_ENTITY

teacher_runner.run_with_params(student_eval_params)

## Done

You now have teacher and student checkpoints plus W&B logs for both phases in one notebook run.